# 10 — Do matched-shop history features help? (optional research experiment)

**Question:** does comparing the same shops in consecutive weeks add predictive value beyond the original features?

After completed 06, select your project kernel and **Run All**. Notebook 09 is a separate sensitivity analysis, not a prerequisite. No path/mode edits, new packages or 01–08 rerun are needed.

Compare **original features** against **original + two matched-shop features**, using the already selected model settings on the same three expanding folds in 2025. Fit preprocessing separately within each training fold. Keep every original label, row and eligibility rule unchanged. Both versions use cutoff **0.5**, as in the original fold comparison; no cutoff or hyperparameter search occurs.

The new features use only the current completed week and its exact previous calendar week. They contain no next-week prices, future coverage, outcomes, external forecasts or retrospective price exclusions. Existing fold-specific cleaning/item policies remain unchanged. Current-week prices are assumed available after that week ends; publication delays/backfills are not modelled.

This is **post-hoc exploratory development**, motivated after earlier results were examined. These folds are not fresh validation, and the original model family was already selected using them. A gain would justify future evaluation, not prove better independent test performance. This notebook does not access the 2026 test file, fit a new dashboard model or alter Notebooks 01–09.

In [1]:
import hashlib
import json
from pathlib import Path
from importlib import reload, metadata
import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.metrics import average_precision_score, precision_score, recall_score, f1_score, confusion_matrix
import pricecatcher_config as cfg
cfg = reload(cfg)
FOLDS = ["fold_1", "fold_2", "fold_3"]
EXTRA = ["matched_shop_change_1w_pct", "matched_shop_count_1w"]
KEYS = cfg.GROUP_KEYS + ["week_start"]

def file_hash(path):
    with Path(path).open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()

model_dir = next((p for p in cfg.MODEL_DIRS if (p / "best_model.joblib").is_file()), None)
if model_dir is None:
    raise FileNotFoundError("Complete Notebook 06 first; keep its original saved experiment.")
manifest = json.loads((model_dir / "model_manifest.json").read_text())
selection = json.loads((model_dir / "selection.json").read_text())
if manifest.get("run_status") != "completed" or selection != manifest["selection"]:
    raise ValueError("A matching completed original model and selection are required.")
if file_hash(model_dir / "best_model.joblib") != manifest["model_sha256"]:
    raise ValueError("Saved original model changed. Restore the matching file.")
packages = ["pandas", "numpy", "scikit-learn", "joblib"]
if selection["family"] == "LightGBM":
    packages.append("lightgbm")
if selection["family"] == "XGBoost":
    packages += [p for p in ["xgboost", "xgboost-cpu"] if p in manifest["package_versions"]]
for package in packages:
    if package in manifest["package_versions"] and metadata.version(package) != manifest["package_versions"][package]:
        raise ValueError(f"Use the original experiment's {package} version for a reproducible comparison.")
source_manifest = cfg.verify_source_snapshot()
paths = [model_dir / name for name in ["best_model.joblib", "model_manifest.json", "selection.json", "fold_metrics.csv"]]
paths += [cfg.STAGE1_DIR / "source_manifest.json"]
fold_settings = {}
for fold in FOLDS:
    run = cfg.PROJECT_ROOT / "experiments" / fold
    folder = run / "05_modelling"
    for name in ["experiment_settings.json", "train.parquet", "validation.parquet"]:
        path = folder / name
        if not path.exists():
            raise FileNotFoundError(f"Missing {fold}/{name}. Restore the original fold preparation from 06.")
        if file_hash(path) != manifest["input_sha256"][f"{fold}/{name}"]:
            raise ValueError(f"Frozen input changed: {fold}/{name}. Restore its original file.")
        paths.append(path)
    settings = json.loads((folder / "experiment_settings.json").read_text())
    cleaning_path = run / "03_cleaned" / "cleaning_summary.json"
    cleaning = json.loads(cleaning_path.read_text())
    if cleaning["preparation"] != settings["preparation"]:
        raise ValueError(f"{fold}: cleaning and frozen modelling policies disagree.")
    if settings["preparation"].get("source_snapshot_sha256") != source_manifest["source_snapshot_sha256"]:
        raise ValueError(f"{fold}: a matching source content snapshot is required for this experiment.")
    if settings["preparation"]["apply_documented_exclusions"] or cleaning["reviewed_records_excluded"]:
        raise ValueError("Early folds must retain retrospective manual exceptions.")
    if tuple(settings[k] for k in ["train_end", "validation_end", "test_end"]) != cfg.EXPERIMENTS[fold]:
        raise ValueError(f"{fold}: dates differ from the declared expanding folds.")
    if settings["feature_columns"] != selection["feature_columns"]:
        raise ValueError(f"{fold}: original feature list differs from the frozen model.")
    paths += [cleaning_path, run / "03_cleaned" / "daily_prices.parquet"]
    fold_settings[fold] = settings
input_hashes = {str(p.relative_to(cfg.PROJECT_ROOT)): file_hash(p) for p in paths}
notebook_path = Path(cfg.__file__).parent / "10_matched_shop_feature_experiment.ipynb"
notebook_code = "\n".join("".join(c["source"]) for c in json.loads(notebook_path.read_text())["cells"] if c["cell_type"] == "code")
protocol = {"schema_version": 1, "status": "post-hoc exploratory expanding-fold feature comparison",
    "candidate": selection["candidate"], "extra_features": EXTRA, "fold_cutoff": 0.5,
    "aggregation": "median of shop-week medians on common shops; current/previous ratio",
    "code_sha256": hashlib.sha256(notebook_code.encode()).hexdigest(), "input_sha256": input_hashes,
    "package_versions": {p: metadata.version(p) for p in packages}}
experiment_id = hashlib.sha256(json.dumps(protocol, sort_keys=True).encode()).hexdigest()[:16]
output_dir = cfg.PROJECT_ROOT / "10_matched_shop_experiment" / experiment_id
output_dir.mkdir(parents=True, exist_ok=True)
print("Source run:", cfg.PROJECT_ROOT)
print("Selected settings:", selection["candidate"], "| New features:", EXTRA)
print("Three 2025 folds only. Outputs:", output_dir)

Source run: /Users/alfredyap/Documents/GitHub/FYP/data/runs/run_20261019_013241_618240
Selected settings: lgb_leaves31 | New features: ['matched_shop_change_1w_pct', 'matched_shop_count_1w']
Three 2025 folds only. Outputs: /Users/alfredyap/Documents/GitHub/FYP/data/runs/run_20261019_013241_618240/10_matched_shop_experiment/dc282de743f4bc3a


## Build two past-only features

For local series and current week t, take one price median per shop. Match shop IDs with week t−1, then calculate:

- `matched_shop_change_1w_pct` = 100 × (median current price on common shops / median previous price on those same shops − 1).
- `matched_shop_count_1w` = number of common shops.

At least one common shop is needed; no common shops means a missing change and count zero. This is not the median of shop-level percentage changes. The feature reduces shop-composition changes in its own comparison; it does not make the original target a fixed-shop price index. Unequal coverage within a shop and unit/source errors can still matter. Missing feature values are imputed from the training fold only.

In [2]:
def matched_features(daily):
    """Current and exact previous week only, with no learned bounds or future shifts."""
    daily = daily.copy()
    daily["week_start"] = daily.date - pd.to_timedelta(daily.date.dt.weekday, unit="D")
    shop = daily.groupby(KEYS + ["premise_code"])["price"].median().rename("current_price").reset_index()
    previous = shop.rename(columns={"current_price": "previous_price"}).copy()
    previous["week_start"] += pd.Timedelta(days=7)
    pairs = shop.merge(previous, on=KEYS + ["premise_code"], how="inner", validate="one_to_one")
    matched = pairs.groupby(KEYS).agg(current_matched_median=("current_price", "median"),
        previous_matched_median=("previous_price", "median"), matched_shop_count_1w=("premise_code", "size")).reset_index()
    matched["matched_shop_change_1w_pct"] = 100 * (matched.current_matched_median / matched.previous_matched_median - 1)
    return matched[KEYS + EXTRA]

def add_features(frame, matched):
    result = frame.merge(matched, on=KEYS, how="left", sort=False, validate="one_to_one")
    result["matched_shop_count_1w"] = result.matched_shop_count_1w.fillna(0).astype(int)
    pd.testing.assert_frame_equal(result[frame.columns], frame.reset_index(drop=True))
    if not np.isfinite(result[EXTRA].fillna(0).to_numpy(dtype=float)).all():
        raise ValueError("Non-finite matched-shop features need source investigation.")
    return result

def inputs(frame, settings, extra):
    X = frame[settings["feature_columns"] + extra].copy()
    for column in settings["categorical_features"]:
        X[column] = X[column].astype("string").fillna("__missing__").astype(str)
    return X

def new_pipeline(template, y, extra):
    pipeline = clone(template)  # Clears fitted state; no December-trained encoding or imputation is reused.
    preprocessing = pipeline.named_steps["preprocess"]
    preprocessing.set_params(transformers=[(name, transformer, columns + extra if name == "numeric" else columns)
        for name, transformer, columns in preprocessing.transformers])
    if selection["family"] == "XGBoost":
        pipeline.set_params(model__scale_pos_weight=(len(y) - y.sum()) / y.sum())
    return pipeline

def metrics(y, scores):
    predicted = scores >= 0.5
    tn, fp, fn, tp = confusion_matrix(y, predicted, labels=[0, 1]).ravel()
    return {"rows": len(y), "positives": int(y.sum()),
        "average_precision": float(average_precision_score(y, scores)) if y.nunique() == 2 else np.nan,
        "precision": float(precision_score(y, predicted, zero_division=0)),
        "recall": float(recall_score(y, predicted, zero_division=0)) if y.sum() else np.nan,
        "f1": float(f1_score(y, predicted, zero_division=0)) if y.sum() else np.nan,
        "true_alerts": int(tp), "false_alerts": int(fp), "missed_surges": int(fn), "true_negatives": int(tn)}

## Fit both versions separately in every fold

The original-feature model must reproduce its saved fold AP within numerical tolerance before a comparison is accepted. Daily aggregation must also reproduce the current-price/coverage features in the frozen frames. These checks help detect changed source/preparation or a mismatched model environment.

Every validation frame is identical for both versions. Match availability is reported instead of excluding rows lacking common shops. There is no manual price exclusion based on the later shortlist from 03/09. No new winning model is automatically selected.

In [3]:
cache_path = output_dir / "experiment_manifest.json"
artifact_names = ["fold_comparison.csv", "fold_summary.csv", "group_comparison.csv",
    "matched_coverage.csv", "validation_predictions.parquet"]
cache = json.loads(cache_path.read_text()) if cache_path.exists() else None
if cache is not None:
    if cache["protocol"] != protocol or set(cache["output_sha256"]) != set(artifact_names):
        raise ValueError("Feature-experiment cache does not match the declared protocol.")
    if any(not (output_dir / name).is_file() or file_hash(output_dir / name) != cache["output_sha256"][name]
           for name in artifact_names):
        raise ValueError("Derived feature-experiment cache changed. Restore its matching artifacts.")
    print("Reusing completed comparison; no model fitting.")
else:
    template = joblib.load(model_dir / "best_model.joblib")
    reference_metrics = pd.read_csv(model_dir / "fold_metrics.csv")
    rows, group_rows, coverage_rows, predictions = [], [], [], []
    for fold, settings in fold_settings.items():
        print("Preparing matched-shop history:", fold)
        run = cfg.PROJECT_ROOT / "experiments" / fold
        end = pd.Timestamp(settings["validation_end"])
        daily = pd.read_parquet(run / "03_cleaned" / "daily_prices.parquet",
            columns=cfg.GROUP_KEYS + ["date", "premise_code", "price"], filters=[("date", "<=", end)])
        daily["date"] = pd.to_datetime(daily.date)
        if daily.date.gt(end).any() or daily.duplicated(["date", "premise_code", "item_code"]).any():
            raise ValueError("Daily history must have unique keys and stay within the fold.")
        if not (np.isfinite(daily.price) & daily.price.gt(0)).all():
            raise ValueError("Daily prices must be finite and positive.")
        daily["week_start"] = daily.date - pd.to_timedelta(daily.date.dt.weekday, unit="D")
        daily = daily.loc[daily.week_start.ge(pd.Timestamp(cfg.START_MONTH + "-01"))
            & (daily.week_start + pd.Timedelta(days=6)).le(end)].copy()
        matched = matched_features(daily)
        check_columns = ["weekly_median_price", "weekly_std_price", "record_count", "premise_count", "observed_days"]
        weekly_check = daily.groupby(KEYS).agg(weekly_median_price=("price", "median"),
            weekly_std_price=("price", "std"), record_count=("price", "size"),
            premise_count=("premise_code", "nunique"), observed_days=("date", "nunique")).reset_index()
        frames = {}
        for split in ["train", "validation"]:
            frame = pd.read_parquet(run / "05_modelling" / f"{split}.parquet")
            if frame.empty or frame.duplicated(KEYS).any() or frame.surge_label.nunique() != 2:
                raise ValueError(f"{fold}/{split}: nonempty unique rows and both classes are required.")
            train_end = pd.Timestamp(settings["train_end"])
            if (split == "train" and frame.next_week_end.gt(train_end).any()) or (split == "validation" and (
                    (frame.week_start + pd.Timedelta(days=6)).le(train_end).any() or frame.next_week_end.gt(end).any())):
                raise ValueError("Fold frames cross chronological cutoffs.")
            audit = frame[KEYS + check_columns].merge(weekly_check, on=KEYS, how="left", validate="one_to_one", suffixes=("_frozen", "_daily"))
            for column in check_columns:
                np.testing.assert_allclose(audit[column + "_frozen"].to_numpy(dtype=float),
                    audit[column + "_daily"].to_numpy(dtype=float), rtol=1e-9, atol=1e-9, equal_nan=True)
            frames[split] = add_features(frame, matched)
            coverage_rows.append({"fold": fold, "split": split, "rows": len(frame),
                "rows_with_matched_shops": int(frames[split].matched_shop_count_1w.gt(0).sum()),
                "rows_without_matched_shops": int(frames[split].matched_shop_count_1w.eq(0).sum()),
                "rows_with_two_or_more_matches": int(frames[split].matched_shop_count_1w.ge(2).sum())})
        train, validation = frames["train"], frames["validation"]
        y_train, y_validation = train.surge_label.astype(int), validation.surge_label.astype(int)
        prediction = validation[KEYS + ["next_week_end", "surge_label"]].copy()
        prediction.insert(0, "fold", fold)
        for version, extra in [("original_features", []), ("with_matched_shop_history", EXTRA)]:
            print("Fitting:", fold, version)
            pipeline = new_pipeline(template, y_train, extra)
            pipeline.fit(inputs(train, settings, extra), y_train)
            scores = pipeline.predict_proba(inputs(validation, settings, extra))[:, 1]
            result = metrics(y_validation, scores)
            if version == "original_features":
                expected = reference_metrics.loc[reference_metrics.fold.eq(fold)
                    & reference_metrics.candidate.eq(selection["candidate"]), "average_precision"]
                if len(expected) != 1 or not np.isclose(result["average_precision"], expected.iloc[0], rtol=1e-6, atol=1e-6):
                    raise ValueError(f"{fold}: original fold AP does not reproduce. Investigate before interpreting improvements.")
            rows.append({"fold": fold, "version": version, "cutoff": 0.5, **result})
            for group, part in validation.groupby("project_group"):
                group_rows.append({"fold": fold, "version": version, "project_group": group,
                    **metrics(part.surge_label.astype(int), scores[part.index])})
            prediction[version + "_score"] = scores
            del pipeline
        predictions.append(prediction)
        del daily, weekly_check, frames, train, validation, matched
    comparison = pd.DataFrame(rows)
    summary = comparison.groupby("version").agg(mean_ap=("average_precision", "mean"),
        std_ap=("average_precision", "std"), mean_f1=("f1", "mean"), folds=("fold", "nunique")).reset_index()
    comparison.to_csv(output_dir / "fold_comparison.csv", index=False)
    summary.to_csv(output_dir / "fold_summary.csv", index=False)
    pd.DataFrame(group_rows).to_csv(output_dir / "group_comparison.csv", index=False)
    pd.DataFrame(coverage_rows).to_csv(output_dir / "matched_coverage.csv", index=False)
    pd.concat(predictions, ignore_index=True).to_parquet(output_dir / "validation_predictions.parquet", index=False)
    if any(file_hash(cfg.PROJECT_ROOT / name) != value for name, value in input_hashes.items()):
        raise ValueError("An original input changed during the comparison; results are incomplete.")
    cache = {"protocol": protocol, "prepared_test_accessed": False, "dashboard_model_replaced": False,
        "output_sha256": {name: file_hash(output_dir / name) for name in artifact_names}}
    temporary = cache_path.with_suffix(".tmp")
    temporary.write_text(json.dumps(cache, indent=2))
    temporary.replace(cache_path)
comparison = pd.read_csv(output_dir / "fold_comparison.csv")
print("Fold results, with identical validation rows and fixed cutoff:")
display(comparison)
display(pd.read_csv(output_dir / "fold_summary.csv"))
changes = comparison.pivot(index="fold", columns="version", values="average_precision")
changes["ap_difference"] = changes.with_matched_shop_history - changes.original_features
display(changes)
display(pd.read_csv(output_dir / "matched_coverage.csv"))
print("Saved:", output_dir)
print("Exploratory feature experiment; main model, cutoff, dashboard inputs and test remain unchanged.")

Preparing matched-shop history: fold_1
Fitting: fold_1 original_features
Fitting: fold_1 with_matched_shop_history
Preparing matched-shop history: fold_2
Fitting: fold_2 original_features
Fitting: fold_2 with_matched_shop_history
Preparing matched-shop history: fold_3
Fitting: fold_3 original_features
Fitting: fold_3 with_matched_shop_history
Fold results, with identical validation rows and fixed cutoff:


,fold,version,cutoff,rows,positives,average_precision,precision,recall,f1,true_alerts,false_alerts,missed_surges,true_negatives
0,fold_1,original_features,0.5,31971,3009,0.285432,0.210886,0.696577,0.323757,2096,7843,913,21119
1,fold_1,with_matched_shop_history,0.5,31971,3009,0.294922,0.207235,0.723496,0.322184,2177,8328,832,20634
2,fold_2,original_features,0.5,31753,3493,0.396019,0.334903,0.631263,0.437630,2205,4379,1288,23881
3,fold_2,with_matched_shop_history,0.5,31753,3493,0.390860,0.339110,0.610650,0.436063,2133,4157,1360,24103
4,fold_3,original_features,0.5,54666,6244,0.385710,0.301117,0.682095,0.417795,4259,9885,1985,38537
5,fold_3,with_matched_shop_history,0.5,54666,6244,0.373290,0.294067,0.653267,0.405568,4079,9792,2165,38630


,version,mean_ap,std_ap,mean_f1,folds
0,original_features,0.355720,0.061089,0.393061,3
1,with_matched_shop_history,0.353024,0.051079,0.387938,3


version,original_features,with_matched_shop_history,ap_difference
fold,,,
fold_1,0.285432,0.294922,0.009490
fold_2,0.396019,0.390860,-0.005158
fold_3,0.385710,0.373290,-0.012420


,fold,split,rows,rows_with_matched_shops,rows_without_matched_shops,rows_with_two_or_more_matches
0,fold_1,train,66384,66292,92,45444
1,fold_1,validation,31971,31923,48,22069
2,fold_2,train,101515,101370,145,69408
3,fold_2,validation,31753,31701,52,23122
4,fold_3,train,138296,138055,241,96134
5,fold_3,validation,54666,54583,83,41355


Saved: /Users/alfredyap/Documents/GitHub/FYP/data/runs/run_20261019_013241_618240/10_matched_shop_experiment/dc282de743f4bc3a
Exploratory feature experiment; main model, cutoff, dashboard inputs and test remain unchanged.


## Read the outcome honestly

Look at AP differences in **each fold**, food-group results, matched-shop availability and false alerts—not just the mean. Three reused folds cannot establish statistical or universal superiority. The added count feature also conveys monitoring support, so a gain does not isolate composition adjustment alone.

If gains are small or inconsistent, retain the simpler original model and report this negative/mixed finding. If gains are consistent and useful, freeze the proposed feature policy before collecting a fresh later evaluation period; do not automatically replace the original test claim or dashboard model. Do not change the original target to improve these scores.

Share `fold_comparison.csv`, `fold_summary.csv`, `group_comparison.csv` and `matched_coverage.csv` for interpretation. Completed matching runs reopen without retraining. No model checkpoint is saved here because this notebook compares development evidence rather than deploying a model.